In [ ]:
import os 
import time 
import random

import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_fscore_support
)

In [ ]:
SEED = 42
EPOCHS = 20
BATCH_SIZE = 128
LEARNING_RATE = 0.001

MNIST_VAL_SIZE = 5000
DIGITS_TEST_SIZE = 0.20
DIGITS_VAL_SIZE = 0.20

OUTPUT_DIR = "output"
RESULT_DIR = os.path.join(OUTPUT_DIR, "results")
FIGURE_DIR = os.path.join(OUTPUT_DIR, "figures")
MODEL_DIR = os.path.join(OUTPUT_DIR, "models")

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

CLASS_NAMES = [
    "0", "1", "2", "3", "4", "5", "6", "7", "8", "9"
]

NUM_CLASSES = len(CLASS_NAMES)

In [ ]:
def load_mnist():
    print("\n" + "="*80)
    print("Loading MNIST dataset...")
    print("="*80)

    (
        X_train_full,
        y_train_full,
    ), (
        X_test,
        y_test,
    ) = tf.keras.datasets.mnist.load_data()

    X_train_full = X_train_full[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    X_train_full = (
        X_train_full.astype("float32") / 255.0
    )

    X_test = (
        X_test.astype("float32") / 255.0
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=MNIST_VAL_SIZE,
        random_state=SEED,
        stratify=y_train_full,
    )

    return (
        X_train, y_train, X_val, y_val, X_test, y_test
    )

In [ ]:
def load_digits_dataset():
    print("\n" + "="*80)
    print("Loading Digits dataset...")
    print("="*80)

    digits = load_digits()

    X = digits.images
    y = digits.target

    X = (
        X.astype("float32") / 16.0
    )

    X = X[..., np.newaxis]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=DIGITS_TEST_SIZE,
        random_state=SEED,
        stratify=y,
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train,
        y_train,
        test_size=DIGITS_VAL_SIZE,
        random_state=SEED,
        stratify=y_train,
    )

    return (
        X_train, y_train, X_val, y_val, X_test, y_test
    )

In [ ]:
datasets = {
    "MNIST": load_mnist(),
    "Digits": load_digits_dataset()
}

for dataset_name, dataset in datasets.items():
    (
        X_train, y_train, X_val, y_val, X_test, y_test
    ) = dataset

    print("\n" + "="*80)
    print(f"{dataset_name} Dataset Summary:")
    print("="*80)
    print(f"Training set shape: {X_train.shape}, {y_train.shape}")
    print(f"Validation set shape: {X_val.shape}, {y_val.shape}")
    print(f"Test set shape: {X_test.shape}, {y_test.shape}")

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(12, 5))

for row, (dataset_name, dataset) in enumerate(datasets.items()):
    X_train = dataset[0]
    y_train = dataset[1]

    for i in range(10):
        axes[row, i].imshow(
            X_train[i].squeeze(), cmap="gray"
        )

        axes[row, i].set_title(str(y_train[i]))
        axes[row, i].axis("off")

    axes[row, 0].set_ylabel(dataset_name, fontsize=14, fontweight="bold")

fig.suptitle("Sample Images from MNIST and Digits Datasets", fontsize=16, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "sample_images.png"),
            dpi=300, bbox_inches="tight")
plt.close()